In [ ]:
import sqlite3
import json
from datetime import datetime

class CoronaDatabase:
    def __init__(self, db_path='corona_detection.db'):
        self.db_path = db_path
        self.init_database()
    
    def init_database(self):
        """Initialize database tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create patients table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS patients (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                age INTEGER NOT NULL,
                gender TEXT NOT NULL,
                phone TEXT,
                address TEXT,
                registration_date TEXT
            )
        ''')
        
        # Create tests table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS tests (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                patient_id INTEGER,
                test_date TEXT,
                temperature REAL,
                cough INTEGER,
                fever INTEGER,
                breathing_difficulty INTEGER,
                loss_of_taste_smell INTEGER,
                fatigue INTEGER,
                chest_pain INTEGER,
                travel_history INTEGER,
                contact_with_infected INTEGER,
                test_result TEXT,
                severity TEXT,
                FOREIGN KEY (patient_id) REFERENCES patients (id)
            )
        ''')
        
        conn.commit()
        conn.close()
    
    def execute_query(self, query, params=()):
        """Execute a query and return results"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute(query, params)
        
        if query.strip().upper().startswith('SELECT'):
            result = cursor.fetchall()
        else:
            conn.commit()
            result = cursor.lastrowid
        
        conn.close()
        return result
    
    def get_patient_stats(self):
        """Get patient statistics"""
        query = "SELECT COUNT(*) FROM patients"
        total_patients = self.execute_query(query)[0][0]
        
        query = "SELECT COUNT(*) FROM tests"
        total_tests = self.execute_query(query)[0][0]
        
        query = "SELECT COUNT(*) FROM tests WHERE test_result = 'Positive'"
        positive_cases = self.execute_query(query)[0][0]
        
        return {
            'total_patients': total_patients,
            'total_tests': total_tests,
            'positive_cases': positive_cases
        }